# **Installing the 'Faker' package which we used for generatung our simulation database**

In [2]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.4 MB/s eta 0:00:00


# **Generating the database with the connections between the tables**

In [17]:
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

YEAR = 2024
NUM_PORTS = 6
NUM_VESSELS = 100
TARGET_AIS = 50000

time_range = pd.date_range(start=f'{YEAR}-01-01', end=f'{YEAR}-12-31 23:00:00', freq='h')
dim_time = pd.DataFrame({
    'time_key': time_range.strftime('%Y%m%d%H').astype(int),
    'date_actual': time_range.date,
    'hour_actual': time_range.hour,
    'day_name': time_range.day_name(),
    'month_name': time_range.month_name(),
    'quarter': time_range.quarter,
    'year': time_range.year,
    'is_holiday': np.random.choice([0, 1], size=len(time_range), p=[0.95, 0.05]),
    'shift_name': np.where(time_range.hour < 8, 'Night', np.where(time_range.hour < 16, 'Morning', 'Day'))
})

df_ports = pd.DataFrame([
    {'port_key': 1, 'port_id': 'EG_ALY', 'latitude': 31.20, 'longitude': 29.91},
    {'port_key': 2, 'port_id': 'EG_PSD', 'latitude': 31.25, 'longitude': 32.28},
    {'port_key': 3, 'port_id': 'EG_DMT', 'latitude': 31.41, 'longitude': 31.81},
    {'port_key': 4, 'port_id': 'EG_SOK', 'latitude': 29.95, 'longitude': 32.35},
    {'port_key': 5, 'port_id': 'EG_SAF', 'latitude': 26.73, 'longitude': 33.93},
    {'port_key': 6, 'port_id': 'EG_SUZ', 'latitude': 29.96, 'longitude': 32.55}
])

vessels = []
for i in range(1, NUM_VESSELS + 1):
    vessels.append({
        'vessel_key': i,
        'vessel_imo': fake.unique.numerify('#######'),
        'vessel_type': random.choice(['Container', 'Tanker', 'Bulk Carrier']),
        'draft_status': round(random.uniform(8.0, 16.0), 2)
    })
dim_vessels = pd.DataFrame(vessels)

full_index = pd.MultiIndex.from_product(
    [dim_time['time_key'], df_ports['port_key']],
    names=['time_key', 'port_key']
)
template_df = pd.DataFrame(index=full_index).reset_index()

fact_port_conditions = template_df.copy()
fact_port_conditions['condition_id'] = range(1, len(fact_port_conditions) + 1)
fact_port_conditions['wind_speed_bft'] = np.random.randint(1, 12, size=len(fact_port_conditions))
fact_port_conditions['wave_height'] = np.round(np.random.uniform(0.1, 4.5, size=len(fact_port_conditions)), 2)
fact_port_conditions['visibility_km'] = np.round(np.random.uniform(1.0, 15.0, size=len(fact_port_conditions)), 2)
fact_port_conditions['port_status'] = (fact_port_conditions['wind_speed_bft'] > 7).astype(int)

fact_port_operations = template_df.copy()
fact_port_operations['op_id'] = range(1, len(fact_port_operations) + 1)
fact_port_operations['berth_occupancy_rate'] = np.round(np.random.uniform(30.0, 95.0, size=len(fact_port_operations)), 2)
fact_port_operations['crane_efficiency'] = np.random.randint(5, 25, size=len(fact_port_operations))
fact_port_operations['customs_time_hours'] = np.round(np.random.uniform(1.0, 24.0, size=len(fact_port_operations)), 2)
fact_port_operations['cargo_category'] = np.random.choice(['Container', 'Bulk', 'Oil'], size=len(fact_port_operations))
fact_port_operations['truck_queue_length'] = np.random.randint(0, 100, size=len(fact_port_operations))

ais_data = []
status_lookup = fact_port_conditions.set_index(['time_key', 'port_key'])['port_status'].to_dict()

for i in range(1, TARGET_AIS + 1):
    t_key = random.choice(dim_time['time_key'])
    p_key = random.randint(1, 6)
    v_key = random.randint(1, 100)

    p_status = status_lookup.get((t_key, p_key), 0)
    wait = round(random.uniform(12, 48), 2) if p_status == 1 else round(random.uniform(0.5, 8), 2)

    ais_data.append({
        'movement_id': f"MOV_{i:05d}",
        'vessel_key': v_key,
        'time_key': t_key,
        'port_key': p_key,
        'current_lat': round(df_ports.loc[p_key-1, 'latitude'] + random.uniform(-0.1, 0.1), 4),
        'current_lon': round(df_ports.loc[p_key-1, 'longitude'] + random.uniform(-0.1, 0.1), 4),
        'speed_knots': round(random.uniform(0, 20), 1),
        'eta_reported': datetime.now() + timedelta(days=1),
        'waiting_time_hours': wait
    })
df_ais = pd.DataFrame(ais_data)

costs_data = []
v_types_map = dim_vessels.set_index('vessel_key')['vessel_type'].to_dict()

for _, row in df_ais.iterrows():
    v_type = v_types_map.get(row['vessel_key'], 'Container')

    demurrage = round(random.uniform(30000, 60000), 2) if v_type in ['Tanker', 'Container'] else round(random.uniform(10000, 25000), 2)

    costs_data.append({
        'cost_id': f"COST_{row['movement_id']}",
        'movement_id': row['movement_id'],
        'route_id': f"R_{random.randint(100, 999)}",
        'vessel_key': row['vessel_key'],
        'time_key': row['time_key'],
        'daily_demurrage_cost': demurrage,
        'fuel_cost_per_km': round(random.uniform(15.0, 45.0), 2),
        'perishability_loss_rate': round(random.uniform(0.0001, 0.0050), 4),
        'alternative_port_cost': round(random.uniform(5000, 20000), 2)
    })

df_costs = pd.DataFrame(costs_data)

master_df = df_ais.merge(dim_vessels, on='vessel_key', how='left')
master_df = master_df.merge(df_ports, on='port_key', how='left')
master_df = master_df.merge(dim_time, on='time_key', how='left')
master_df = master_df.merge(fact_port_conditions, on=['time_key', 'port_key'], how='left')
master_df = master_df.merge(fact_port_operations, on=['time_key', 'port_key'], how='left')
master_df = master_df.merge(df_costs.drop(columns=['vessel_key', 'time_key']), on='movement_id', how='left')

print(f"total number of observations {len(master_df)}")
print(f"missing values {master_df.isnull().sum().sum()}")
print(f"the columns: \n{master_df.columns.tolist()}")

total number of observations 50000
missing values 0
the columns: 
['movement_id', 'vessel_key', 'time_key', 'port_key', 'current_lat', 'current_lon', 'speed_knots', 'eta_reported', 'waiting_time_hours', 'vessel_imo', 'vessel_type', 'draft_status', 'port_id', 'latitude', 'longitude', 'date_actual', 'hour_actual', 'day_name', 'month_name', 'quarter', 'year', 'is_holiday', 'shift_name', 'condition_id', 'wind_speed_bft', 'wave_height', 'visibility_km', 'port_status', 'op_id', 'berth_occupancy_rate', 'crane_efficiency', 'customs_time_hours', 'cargo_category', 'truck_queue_length', 'cost_id', 'route_id', 'daily_demurrage_cost', 'fuel_cost_per_km', 'perishability_loss_rate', 'alternative_port_cost']


# **Checking the Database**

In [18]:
d = [dim_time,
     df_ports,
     dim_vessels,
     fact_port_conditions,
     fact_port_operations,
     df_ais]

for table in d:
  table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   time_key     8784 non-null   int64 
 1   date_actual  8784 non-null   object
 2   hour_actual  8784 non-null   int32 
 3   day_name     8784 non-null   object
 4   month_name   8784 non-null   object
 5   quarter      8784 non-null   int32 
 6   year         8784 non-null   int32 
 7   is_holiday   8784 non-null   int64 
 8   shift_name   8784 non-null   object
dtypes: int32(3), int64(2), object(4)
memory usage: 514.8+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   port_key   6 non-null      int64  
 1   port_id    6 non-null      object 
 2   latitude   6 non-null      float64
 3   longitude  6 non-null      float64
dtypes: float64(2), int64(1), object(1)
m

# **Checking the master table**

In [9]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 40 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   movement_id              50000 non-null  object        
 1   vessel_key               50000 non-null  int64         
 2   time_key                 50000 non-null  int64         
 3   port_key                 50000 non-null  int64         
 4   current_lat              50000 non-null  float64       
 5   current_lon              50000 non-null  float64       
 6   speed_knots              50000 non-null  float64       
 7   eta_reported             50000 non-null  datetime64[ns]
 8   waiting_time_hours       50000 non-null  float64       
 9   vessel_imo               50000 non-null  object        
 10  vessel_type              50000 non-null  object        
 11  draft_status             50000 non-null  float64       
 12  port_id                  50000 n

# **Saving all the files**

In [10]:
master_df.to_csv('master_date.csv', index=False)

In [11]:
dim_vessels.to_csv('dim_vessels.csv', index=False)

In [12]:
df_ports.to_csv('df_ports.csv', index=False)

In [13]:
dim_time.to_csv('dim_time.csv', index=False)

In [14]:
fact_port_conditions.to_csv('fact_port_conditions.csv', index=False)

In [15]:
fact_port_operations.to_csv('fact_port_operations.csv', index=False)

In [16]:
df_costs.to_csv('df_costs.csv', index=False)